## Libraries and functions

In [1]:
import sys
from pathlib import Path

# Add src/ to sys.path regardless of where Jupyter is launched from
for _candidate in [Path().resolve().parent / "src", Path().resolve() / "src"]:
    if _candidate.exists() and str(_candidate) not in sys.path:
        sys.path.insert(0, str(_candidate))
        print(f"Added to sys.path: {_candidate}")
        break

Added to sys.path: C:\Users\loren\Documents\Postdoc\Compressed_sensing\compressed_sensing_bioacoustics\src


In [ ]:
from compress import Compression

import psutil
import time
import threading
from codecarbon import EmissionsTracker
import datetime
import pandas as pd
import gc
# Run garbage collection
gc.collect()

0

In [9]:
def monitor_memory(stop_event, mem_usage, sampling_interval=0.1):
    """
    Monitor the memory usage of the current process at regular intervals.
    """
    process = psutil.Process(os.getpid())
    while not stop_event.is_set():
        mem = process.memory_info().rss / (1024 ** 2)  # Convert bytes to MB
        mem_usage.append(mem)
        time.sleep(sampling_interval)

In [10]:
def monitor_resources(stop_event, cpu_usage, mem_usage, sampling_interval=0.1):
    """
    Monitor CPU and memory usage of the current process at regular intervals.
    - cpu_usage: list to store CPU usage (%) over time
    - mem_usage: list to store memory usage (MB) over time
    """
    process = psutil.Process(os.getpid())
    
    while not stop_event.is_set():
        # CPU percent since last call (per process)
        cpu = process.cpu_percent(interval=None)
        # Memory usage (RSS in MB)
        mem = process.memory_info().rss / (1024 ** 2)
        
        cpu_usage.append(cpu)
        mem_usage.append(mem)
        
        time.sleep(sampling_interval)

## Compression

In [ ]:
#parameters
folder_audio=r"c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\Thyolo\Audio"
folder_compress=r"c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\Thyolo\Compressed_Audio"
folder_tracking=r"c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\Thyolo\tracking"
converter_path="c:/Users/loren/Documents/Postdoc/Compressed_sensing/ffmpeg-master-latest-win64-gpl-shared/ffmpeg-master-latest-win64-gpl-shared/bin/ffmpeg.exe"

method_compression="aac"  #between mp3, acc, ogg, flac
parameter_compression="96k" #for mp3 ; 96k/192k/320k, flac;0/6/12, ogg:0/5/10  , aac; 96k/192k/320k

emissions_file = Path(folder_tracking) / f"emissions_output_{method_compression}_{parameter_compression}.csv"

In [ ]:
# Initialize compression class
compression = Compression(folder_audio, folder_compress, method_compression, parameter_compression ,converter_path)


In [ ]:
# Lists to store metrics
cpu_usage, mem_usage = [], []
sampling_interval = 0.1  # Adjust as needed
stop_event = threading.Event()


# Start resource monitoring in background
monitor_thread = threading.Thread(target=monitor_resources, args=(stop_event,cpu_usage, mem_usage, sampling_interval))
init_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-5]
print(init_time)
monitor_thread.start()

# Start emissions tracking
tracker = EmissionsTracker(output_file=str(emissions_file))
tracker.start()

# Run compression
starting = time.time()
times=compression.compress()
end=time.time()
execution_time_baseline = time.time() - starting


# Stop trackers
tracker.stop()
stop_event.set()
monitor_thread.join()



print(execution_time_baseline)

## Plot compression metrics evolution

In [ ]:
# Compute summary statistics
avg_cpu = sum(cpu_usage) / len(cpu_usage)
peak_mem = max(mem_usage)

print(f"Average CPU usage: {avg_cpu:.2f}%")
print(f"Peak memory usage: {peak_mem:.2f} MB")
#print(f"Elapsed time: {end - starting :.2f} s")

# Calculate memory usage statistics during prediction
if mem_usage:
    min_mem = min(mem_usage)
    max_mem = max(mem_usage)
    avg_mem = sum(mem_usage) / len(mem_usage)
    print(f"Memory usage during prediction:")
    print(f"Minimum: {min_mem:.4f} MB")
    print(f"Maximum: {max_mem:.4f} MB")
    print(f"Average: {avg_mem :.4f} MB")
else:
    print("No memory usage data collected.")

In [ ]:
#create time_chain dataset with time for each ram value
start_time_str = init_time
length = len(mem_usage)
sampling_interval = 0.1
# Convert the starting time string to a datetime object
start_time = datetime.datetime.strptime(start_time_str, '%Y-%m-%d %H:%M:%S.%f')
# Create a list of timestamps based on the starting time, length, and sampling interval
time_chain = [start_time + datetime.timedelta(seconds=i * sampling_interval) for i in range(length)]
time_chain_str = [time.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3] for time in time_chain]

time_chain_df = pd.DataFrame({'Timestamp': time_chain_str, 'Memory usage': mem_usage, 'CPU usage':cpu_usage })

In [ ]:

time_objects = [datetime.datetime.strptime(ts, '%Y-%m-%d %H:%M:%S.%f') for ts in times]
events_df=pd.DataFrame({'Timestamp': time_objects, 'event': [f for f in os.listdir(folder_audio) if f.endswith(".WAV") or f.endswith(".wav")]})


time_chain_df['Timestamp'] = pd.to_datetime(time_chain_df['Timestamp'])
events_df['Timestamp'] = pd.to_datetime(events_df['Timestamp'])
merged_df=pd.merge(time_chain_df, events_df, on='Timestamp', how='left', sort=True)
merged_df.to_csv(f'{folder_tracking}/Ram_usage_{method_compression}_{parameter_compression}.csv')

In [ ]:
save_path=f'{folder_tracking}/time_execution_{method_compression}_{parameter_compression}.txt'
with open(save_path, "w") as f:
    f.write(f"time execution: {execution_time_baseline}")